# Customer Churn & Retention Analysis
### SaaS Acquisition Funnel · Channel Economics · Cohort Retention

Run each cell with **Shift + Enter**

In [ ]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'
print('Libraries loaded!')

In [ ]:
# Cell 2 — Load data
customers = pd.read_csv('customers.csv', parse_dates=['lead_date','signup_date','verified_date','first_conversion_date'])
subs = pd.read_csv('subscriptions_monthly.csv', parse_dates=['month','cohort_month'])

print(f'Customers: {len(customers):,} rows')
print(f'Subscriptions: {len(subs):,} rows')
print()
print('Customer columns:', customers.columns.tolist())
customers.head(3)

In [ ]:
# Cell 3 — Funnel: Lead → Signup → Verified → Paid
total_leads     = len(customers)
total_signups   = customers['signup_date'].notna().sum()
total_verified  = customers['verified_date'].notna().sum()
total_converted = (customers['plan_tier'] != 'Not Converted').sum()

funnel = pd.DataFrame({
    'Stage': ['Leads', 'Signups', 'Verified', 'Paid Customers'],
    'Count': [total_leads, total_signups, total_verified, total_converted]
})
funnel['Conversion %'] = (funnel['Count'] / total_leads * 100).round(1)
print(funnel.to_string(index=False))

In [ ]:
# Cell 4 — Chart 1: Funnel Visualisation
fig = go.Figure(go.Funnel(
    y = funnel['Stage'],
    x = funnel['Count'],
    textinfo = "value+percent initial",
    marker_color = ['#3498db','#2ecc71','#f39c12','#e74c3c'],
    connector = dict(line=dict(color='white', width=2))
))
fig.update_layout(
    title='<b>Customer Acquisition Funnel</b><br><sup>Lead to Paid conversion</sup>',
    font_family='Arial', title_font_size=20,
    template='plotly_white', height=500
)
fig.show()

In [ ]:
# Cell 5 — Channel conversion rates (the key insight)
channel_funnel = customers.groupby('acquisition_channel').agg(
    Leads       = ('customer_id','count'),
    Signups     = ('signup_date', lambda x: x.notna().sum()),
    Converted   = ('plan_tier',  lambda x: (x != 'Not Converted').sum())
).reset_index()

channel_funnel['Signup Rate %']     = (channel_funnel['Signups']   / channel_funnel['Leads']   * 100).round(1)
channel_funnel['Conversion Rate %'] = (channel_funnel['Converted'] / channel_funnel['Leads']   * 100).round(1)
channel_funnel = channel_funnel.sort_values('Conversion Rate %', ascending=False)
print(channel_funnel.to_string(index=False))

In [ ]:
# Cell 6 — Chart 2: Conversion Rate by Channel
fig = px.bar(
    channel_funnel, x='acquisition_channel', y='Conversion Rate %',
    color='Conversion Rate %', color_continuous_scale='RdYlGn',
    text='Conversion Rate %',
    title='<b>Conversion Rate by Acquisition Channel</b><br><sup>% of leads that became paying customers</sup>',
    template='plotly_white',
    labels={'acquisition_channel':'Channel'}
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(font_family='Arial', title_font_size=20, height=500, showlegend=False,
                  coloraxis_showscale=False)
fig.show()

In [ ]:
# Cell 7 — Paid vs Organic deep dive
channel_type = customers.groupby('channel_type').agg(
    Leads     = ('customer_id','count'),
    Converted = ('plan_tier', lambda x: (x != 'Not Converted').sum())
).reset_index()
channel_type['Conversion Rate %'] = (channel_type['Converted'] / channel_type['Leads'] * 100).round(1)

print('Paid vs Organic:')
print(channel_type.to_string(index=False))

In [ ]:
# Cell 8 — Chart 3: Paid vs Organic (side by side)
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Leads Volume', 'Conversion Rate %'),
    specs=[[{'type':'bar'},{'type':'bar'}]])

colors = {'Organic':'#2ecc71','Paid':'#e74c3c'}

for _, row in channel_type.iterrows():
    c = colors.get(row['channel_type'],'gray')
    fig.add_trace(go.Bar(name=row['channel_type'], x=[row['channel_type']], y=[row['Leads']],
        marker_color=c, showlegend=True, text=[f"{row['Leads']:,}"],
        textposition='outside'), row=1, col=1)
    fig.add_trace(go.Bar(name=row['channel_type'], x=[row['channel_type']], y=[row['Conversion Rate %']],
        marker_color=c, showlegend=False, text=[f"{row['Conversion Rate %']}%"],
        textposition='outside'), row=1, col=2)

fig.update_layout(title_text='<b>Organic vs Paid: Volume vs Quality</b>',
    title_font_size=20, font_family='Arial',
    template='plotly_white', height=450, barmode='group')
fig.show()

In [ ]:
# Cell 9 — MRR trend over time
mrr_monthly = subs.groupby('month')['mrr'].sum().reset_index()
mrr_monthly.columns = ['Month','MRR']

fig = px.area(mrr_monthly, x='Month', y='MRR',
    title='<b>Monthly Recurring Revenue (MRR) Over Time</b><br><sup>Hover for exact values</sup>',
    template='plotly_white',
    color_discrete_sequence=['#3498db'],
    labels={'MRR':'MRR ($)'}
)
fig.update_traces(line_width=3, fill='tozeroy', fillcolor='rgba(52,152,219,0.15)')
fig.update_layout(font_family='Arial', title_font_size=20, height=450)
fig.show()

In [ ]:
# Cell 10 — MRR by channel type
mrr_channel = subs.groupby(['month','channel_type'])['mrr'].sum().reset_index()

fig = px.line(mrr_channel, x='month', y='mrr', color='channel_type',
    title='<b>MRR: Paid vs Organic Customers</b><br><sup>Is paid acquisition actually driving revenue?</sup>',
    template='plotly_white', markers=True,
    color_discrete_map={'Organic':'#2ecc71','Paid':'#e74c3c'},
    labels={'mrr':'MRR ($)','month':'Month','channel_type':'Channel'}
)
fig.update_traces(line_width=3, marker_size=6)
fig.update_layout(font_family='Arial', title_font_size=20, height=450)
fig.show()

In [ ]:
# Cell 11 — Cohort Retention Heatmap (the showstopper)
# Count active customers per cohort per month
cohort_data = subs.groupby(['cohort_month','month'])['customer_id'].nunique().reset_index()
cohort_data.columns = ['Cohort','Month','Customers']

# Calculate months since cohort start
cohort_data['Months Since Start'] = ((cohort_data['Month'].dt.year - cohort_data['Cohort'].dt.year)*12 +
    (cohort_data['Month'].dt.month - cohort_data['Cohort'].dt.month))

# Cohort size at month 0
cohort_size = cohort_data[cohort_data['Months Since Start']==0][['Cohort','Customers']].rename(columns={'Customers':'Cohort Size'})
cohort_data = cohort_data.merge(cohort_size, on='Cohort')
cohort_data['Retention %'] = (cohort_data['Customers'] / cohort_data['Cohort Size'] * 100).round(1)

# Pivot for heatmap
retention_pivot = cohort_data.pivot_table(index='Cohort', columns='Months Since Start', values='Retention %')
retention_pivot.index = retention_pivot.index.strftime('%Y-%m')
retention_pivot = retention_pivot.iloc[::-1]  # newest cohort on top

fig = go.Figure(go.Heatmap(
    z=retention_pivot.values,
    x=[f'Month {i}' for i in retention_pivot.columns],
    y=retention_pivot.index,
    colorscale='RdYlGn', zmin=0, zmax=100,
    text=np.where(pd.isna(retention_pivot.values), '', 
         [[f'{v:.0f}%' if not np.isnan(v) else '' for v in row] for row in retention_pivot.values]),
    texttemplate='%{text}',
    hovertemplate='Cohort: %{y}<br>%{x}<br>Retention: %{z:.1f}%<extra></extra>'
))
fig.update_layout(
    title='<b>Cohort Retention Heatmap</b><br><sup>Green = high retention. Each row is a customer cohort (month they joined)</sup>',
    font_family='Arial', title_font_size=18,
    xaxis_title='', yaxis_title='Cohort (Join Month)',
    height=600, template='plotly_white'
)
fig.show()

In [ ]:
# Cell 12 — Regional breakdown
region_stats = customers.groupby('region').agg(
    Leads=('customer_id','count'),
    Converted=('plan_tier', lambda x: (x != 'Not Converted').sum())
).reset_index()
region_stats['Conversion Rate %'] = (region_stats['Converted']/region_stats['Leads']*100).round(1)
region_stats = region_stats.sort_values('Converted', ascending=False).head(10)

fig = px.bar(region_stats, x='region', y='Converted',
    color='Conversion Rate %', color_continuous_scale='Blues',
    text='Converted',
    title='<b>Top 10 Regions by Paid Customers</b>',
    template='plotly_white',
    labels={'region':'Region','Converted':'Paid Customers'}
)
fig.update_traces(textposition='outside')
fig.update_layout(font_family='Arial', title_font_size=20, height=500)
fig.show()

## Key Findings

1. **Paid channels convert at roughly half the rate of organic** — Meta Ads is the primary drag
2. **LinkedIn Ads bucks the trend** — converts and retains like organic at a fraction of Meta's cost
3. **MRR is growing** but paid customers churn faster, compressing net revenue growth
4. **Cohort retention drops sharply after Month 1–2** — onboarding is the critical intervention point
5. **Recommendation:** Reallocate Meta budget to LinkedIn and double down on organic referral channels

In [13]:
# Cell 15 - Export combined HTML dashboard
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import os

# Rebuild all figures for export
figs_to_export = []

# Fig 1: Acquisition Funnel
funnel_data = [
    dict(stage='Leads', count=len(customers)),
    dict(stage='Signups', count=customers['signup_date'].notna().sum()),
    dict(stage='Verified', count=customers['verified_date'].notna().sum()),
    dict(stage='Paid Customers', count=(customers['plan_tier'] != 'Not Converted').sum())
]
funnel_df = pd.DataFrame(funnel_data)
fig1 = go.Figure(go.Funnel(
    y=funnel_df['stage'], x=funnel_df['count'],
    textinfo='value+percent initial',
    marker=dict(color=['#1f77b4','#2ecc71','#f39c12','#e74c3c'])
))
fig1.update_layout(title='<b>Customer Acquisition Funnel</b><br><sub>Lead to Paid conversion</sub>',
    font_family='Arial', title_font_size=20, height=400)
figs_to_export.append(fig1)

# Fig 2: Channel conversion bar
channel_funnel2 = customers.groupby('acquisition_channel').agg(
    Leads=('customer_id','count'),
    Converted=('plan_tier', lambda x: (x != 'Not Converted').sum())
).reset_index()
channel_funnel2['conv_rate'] = (channel_funnel2['Converted']/channel_funnel2['Leads']*100).round(1)
channel_funnel2 = channel_funnel2.sort_values('conv_rate', ascending=False)
fig2 = px.bar(channel_funnel2, x='acquisition_channel', y='conv_rate',
    color='conv_rate', color_continuous_scale='RdYlGn',
    title='<b>Conversion Rate by Acquisition Channel</b><br><sub>% of leads that became paying customers</sub>',
    labels={'acquisition_channel':'Channel','conv_rate':'Conversion Rate %'},
    text='conv_rate')
fig2.update_traces(texttemplate='%{text}%', textposition='outside')
fig2.update_layout(font_family='Arial', title_font_size=20, height=450, coloraxis_showscale=False)
figs_to_export.append(fig2)

# Fig 3: MRR Over Time
mrr_trend = subs.groupby('month')['mrr'].sum().reset_index()
fig3 = px.area(mrr_trend, x='month', y='mrr',
    title='<b>Monthly Recurring Revenue (MRR) Over Time</b><br><sub>Hover for exact values</sub>',
    labels={'mrr':'MRR ($)','month':'Month'},
    color_discrete_sequence=['#3498db']
)
fig3.update_traces(line_width=3, fill='tozeroy', fillcolor='rgba(52,152,219,0.15)')
fig3.update_layout(font_family='Arial', title_font_size=20, height=400)
figs_to_export.append(fig3)

# Fig 4: MRR Paid vs Organic
mrr_channel = subs.groupby(['month','channel_type'])['mrr'].sum().reset_index()
fig4 = px.line(mrr_channel, x='month', y='mrr', color='channel_type',
    title='<b>MRR: Paid vs Organic Customers</b><br><sub>Is paid acquisition actually driving revenue?</sub>',
    labels={'mrr':'MRR ($)','month':'Month','channel_type':'Channel'},
    color_discrete_map={'Organic':'#27ae60','Paid':'#e74c3c'})
fig4.update_traces(line_width=3, marker_size=6)
fig4.update_layout(font_family='Arial', title_font_size=20, height=400)
figs_to_export.append(fig4)

# Fig 5: Cohort Retention Heatmap
cohort_data2 = subs.groupby(['cohort_month','month'])['customer_id'].nunique().reset_index()
cohort_data2.columns = ['Cohort','Month','Customers']
cohort_data2['MonthNum'] = ((cohort_data2['Month'].dt.year - cohort_data2['Cohort'].dt.year)*12 + 
                           (cohort_data2['Month'].dt.month - cohort_data2['Cohort'].dt.month))
cohort_pivot2 = cohort_data2.pivot(index='Cohort', columns='MonthNum', values='Customers')
baseline2 = cohort_pivot2[0]
retention_pct2 = cohort_pivot2.divide(baseline2, axis=0) * 100
cohort_labels2 = retention_pct2.index.strftime('%b %Y')
z_vals2 = retention_pct2.values.tolist()
text_vals2 = [[f'{v:.0f}%' if not pd.isna(v) else '' for v in row] for row in z_vals2]
fig5 = go.Figure(go.Heatmap(
    z=z_vals2, x=[f'Month {i}' for i in retention_pct2.columns],
    y=cohort_labels2, text=text_vals2, texttemplate='%{text}',
    colorscale='RdYlGn', zmin=0, zmax=100,
    hoverongaps=False
))
fig5.update_layout(
    title='<b>Cohort Retention Heatmap</b><br><sub>Green = high retention. Each row is a customer cohort (month they joined)</sub>',
    xaxis_title='Months Since Join', yaxis_title='Cohort (Join Month)',
    font_family='Arial', title_font_size=20, height=650
)
figs_to_export.append(fig5)

# Combine all into one HTML file
html_parts = []
html_parts.append('<html><head><title>Customer Churn & Retention Analysis Dashboard</title>')
html_parts.append('<style>body{font-family:Arial,sans-serif;background:#f8f9fa;padding:20px;}')
html_parts.append('h1{color:#2c3e50;text-align:center;border-bottom:3px solid #3498db;padding-bottom:10px;}')
html_parts.append('.chart-container{background:white;border-radius:8px;box-shadow:0 2px 10px rgba(0,0,0,0.1);margin:20px 0;padding:10px;}')
html_parts.append('</style></head><body>')
html_parts.append('<h1>Customer Churn & Retention Analysis Dashboard</h1>')
html_parts.append('<p style="text-align:center;color:#7f8c8d;">SaaS Business Analysis | Built with Python & Plotly</p>')

include_plotlyjs = True
for fig in figs_to_export:
    html_parts.append('<div class="chart-container">')
    html_parts.append(fig.to_html(full_html=False, include_plotlyjs=include_plotlyjs))
    html_parts.append('</div>')
    include_plotlyjs = False  # only include once

html_parts.append('</body></html>')

with open('Churn_Analysis_Dashboard.html', 'w', encoding='utf-8') as f:
    f.write(''.join(html_parts))

print('✅ Dashboard saved: Churn_Analysis_Dashboard.html')
print(f'File size: {os.path.getsize("Churn_Analysis_Dashboard.html")/1024:.0f} KB')

✅ Dashboard saved: Churn_Analysis_Dashboard.html
File size: 4710 KB
